# Entrega 1 — Exploración reproducible de LoveDA

**Asignatura:** Visión por Computador  
**Integrantes:** Alvaro Toro Herrera y Jorge Butto Cumsille  
**Tema 3:** LoveDA — Land-Cover Semantic Segmentation

## 1. Objetivo y alcance

Este notebook audita el dataset real, construye un índice lógico, calcula estadísticas descriptivas y prepara el diseño experimental de las siguientes entregas. No implementa ni entrena un modelo, no crea predicciones y no calcula métricas de modelo. LoveDA es segmentación semántica multiclase: el valor **0 es no-data/ignore**, no background ni una octava clase. Como contexto, la documentación oficial informa imágenes de aproximadamente 0,3 m de resolución espacial de Nanjing, Changzhou y Wuhan, dos dominios (Urban/Rural), y desafíos de objetos multiescala, fondos complejos y distribuciones de clase diferentes. Las cifras comprobables se obtienen de los archivos, no se hardcodean como resultados.

## 2. Configuración y reproducibilidad

Todas las rutas son relativas o configurables con `LOVEDA_ROOT`. Las selecciones visuales usan una semilla fija. El análisis procesa un archivo por vez y no guarda imágenes completas en DataFrames.

In [1]:
import os
from pathlib import Path
from IPython.display import display

import pandas as pd

from loveda_eda import (
    CLASS_NAMES as MODULE_CLASS_NAMES,
    IGNORE_INDEX as MODULE_IGNORE_INDEX,
    download_official_loveda,
    experimental_design_text,
    find_dataset_root,
    print_automatic_summary,
    run_full_analysis,
    summarized_tree,
    validate_official_downloads,
)

RANDOM_SEED = 42
DATASET_ROOT = Path(os.environ.get("LOVEDA_ROOT", "data/LoveDA"))
OUTPUT_DIR = Path("outputs")
FIGURES_DIR = OUTPUT_DIR / "figures"
IGNORE_INDEX = 0
CLASS_NAMES = {
    1: "background", 2: "building", 3: "road", 4: "water",
    5: "barren", 6: "forest", 7: "agriculture",
}
assert IGNORE_INDEX == MODULE_IGNORE_INDEX and CLASS_NAMES == MODULE_CLASS_NAMES
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 50)
print(f"DATASET_ROOT configurado: {DATASET_ROOT}")
print(f"RANDOM_SEED: {RANDOM_SEED}; IGNORE_INDEX: {IGNORE_INDEX}")

DATASET_ROOT configurado: data\LoveDA
RANDOM_SEED: 42; IGNORE_INDEX: 0


## 3. Obtención o localización del dataset

Fuente prioritaria: [Zenodo, DOI 10.5281/zenodo.5706578](https://doi.org/10.5281/zenodo.5706578), enlazada por el [repositorio oficial de los autores](https://github.com/Junjue-Wang/LoveDA). Zenodo publica `Train.zip`, `Val.zip`, `Test.zip` y `Datasheet.pdf`, con tamaños y MD5.

Por seguridad, la descarga automática está desactivada por defecto: ocupa aproximadamente 9,6 GB comprimidos. Para activarla se puede definir `LOVEDA_DOWNLOAD=1`. La función evita redescargar artefactos válidos, verifica tamaño y MD5 y extrae bajo `data/LoveDA`. También se puede extraer manualmente y definir `LOVEDA_ROOT`.

In [2]:
AUTO_DOWNLOAD = os.environ.get("LOVEDA_DOWNLOAD", "0") == "1"
if AUTO_DOWNLOAD:
    download_validation_df = download_official_loveda(DATASET_ROOT)
elif Path("data/archives").exists():
    download_validation_df = validate_official_downloads(Path("data/archives"))
else:
    download_validation_df = pd.DataFrame()

if not download_validation_df.empty:
    display(download_validation_df)

try:
    RESOLVED_DATASET_ROOT = find_dataset_root(DATASET_ROOT)
    DATASET_AVAILABLE = True
    print(f"Dataset localizado en: {DATASET_ROOT}")
except FileNotFoundError as exc:
    DATASET_AVAILABLE = False
    RESOLVED_DATASET_ROOT = None
    print(exc)
    print("El notebook continuará sin inventar resultados. Configure LOVEDA_ROOT o active LOVEDA_DOWNLOAD=1.")

,file,exists,expected_size_bytes,actual_size_bytes,size_ok,expected_md5,actual_md5,md5_ok
0,Train.zip,True,4021669263,4021669263,True,de2b196043ed9b4af1690b3f9a7d558f,de2b196043ed9b4af1690b3f9a7d558f,True
1,Val.zip,True,2425958254,2425958254,True,84cae2577468ff0b5386758bb386d31d,84cae2577468ff0b5386758bb386d31d,True
2,Test.zip,True,3126023212,3126023212,True,a489be0090465e01fb067795d24e6b47,a489be0090465e01fb067795d24e6b47,True
3,Datasheet.pdf,True,828005,828005,True,7e0a6434f78c240cbfb18afab87404aa,7e0a6434f78c240cbfb18afab87404aa,True


Dataset localizado en: data\LoveDA


## 4. Inspección de la estructura real

Se detectan directorios, splits, dominios, extensiones y archivos auxiliares desde el sistema de archivos. El árbol limita la profundidad y resume extensiones para no imprimir miles de nombres.

In [3]:
if DATASET_AVAILABLE:
    print(summarized_tree(RESOLVED_DATASET_ROOT))
else:
    print("Estructura real: No disponible porque el dataset no fue localizado.")

LoveDA/
    Test/
        Rural/
            images_png/ (976 archivos)
        Urban/
            images_png/ (820 archivos)
    Train/
        Rural/
            images_png/ (1366 archivos)
            masks_png/ (1366 archivos)
        Urban/
            images_png/ (1156 archivos)
            masks_png/ (1156 archivos)
    Val/
        Rural/
            images_png/ (992 archivos)
            masks_png/ (992 archivos)
        Urban/
            images_png/ (677 archivos)
            masks_png/ (677 archivos)


## 5–7. Lectura, índice y control de calidad

Las imágenes se validan con Pillow y se registran dimensiones, modo, canales, formato, extensión, tamaño y SHA-256. Las máscaras se abren **sin conversión a RGB**, se exige un mapa entero 2D y los conteos se obtienen de los valores observados. Se comprueban emparejamientos, tamaños, etiquetas, corrupción, máscaras all-ignore, huérfanas y faltantes. SHA-256 detecta solo igualdad exacta byte a byte.

In [4]:
analysis = None
if DATASET_AVAILABLE:
    analysis = run_full_analysis(
        dataset_root=RESOLVED_DATASET_ROOT,
        output_dir=OUTPUT_DIR,
        figures_dir=FIGURES_DIR,
        random_seed=RANDOM_SEED,
    )
    dataset_index_df = analysis["dataset_index_df"]
    dataset_issues_df = analysis["dataset_issues_df"]
    print(f"Índice construido: {len(dataset_index_df):,} imágenes")
    display(dataset_index_df.head())
    display(dataset_issues_df["issue_type"].value_counts().rename_axis("issue_type").to_frame("count"))
else:
    print("Análisis omitido: no hay archivos reales que procesar.")

Estructura real resumida:



LoveDA/
    Test/
        Rural/
            images_png/ (976 archivos)
        Urban/
            images_png/ (820 archivos)
    Train/
        Rural/
            images_png/ (1366 archivos)
            masks_png/ (1366 archivos)
        Urban/
            images_png/ (1156 archivos)
            masks_png/ (1156 archivos)
    Val/
        Rural/
            images_png/ (992 archivos)
            masks_png/ (992 archivos)
        Urban/
            images_png/ (677 archivos)
            masks_png/ (677 archivos)


Inspección de imágenes y máscaras:   0%|          | 0/5987 [00:00<?, ?it/s]

Inspección de imágenes y máscaras:   3%|▎         | 193/5987 [00:01<00:30, 192.36it/s]

Inspección de imágenes y máscaras:   7%|▋         | 394/5987 [00:02<00:28, 196.32it/s]

...

Inspección de imágenes y máscaras:  99%|█████████▉| 5934/5987 [01:01<00:00, 79.62it/s]

Inspección de imágenes y máscaras: 100%|██████████| 5987/5987 [01:01<00:00, 97.09it/s]

Índice construido: 5,987 imágenes


,image_id,split,domain,image_path,mask_path,has_mask,width,height,channels,image_mode,image_format,image_extension,mask_mode,mask_format,mask_extension,image_file_size_bytes,mask_file_size_bytes,image_valid,mask_valid,size_match,mask_total_pixels,valid_pixel_count,num_valid_classes_present,ignore_pixel_count,ignore_ratio,observed_labels,sha256_image,sha256_mask,class_1_pixel_count,class_2_pixel_count,class_3_pixel_count,class_4_pixel_count,class_5_pixel_count,class_6_pixel_count,class_7_pixel_count,duplicate_group_id,cross_split_duplicate
0,Test/Rural/4191,Test,Rural,Test/Rural/images_png/4191.png,None,False,1024,1024,3,RGB,PNG,.png,None,None,None,918245,NaN,True,False,None,NaN,NaN,NaN,NaN,NaN,None,6dbef546893aaf2ed9c9c3ee8a0f1b565b0f3a21526745...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,False
1,Test/Rural/4192,Test,Rural,Test/Rural/images_png/4192.png,None,False,1024,1024,3,RGB,PNG,.png,None,None,None,1145041,NaN,True,False,None,NaN,NaN,NaN,NaN,NaN,None,2d7b5e5dfa7f0c379179b3e6737409f8c65f19232816c6...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,False
2,Test/Rural/4193,Test,Rural,Test/Rural/images_png/4193.png,None,False,1024,1024,3,RGB,PNG,.png,None,None,None,1809613,NaN,True,False,None,NaN,NaN,NaN,NaN,NaN,None,08c8475b9e9a9136bd864f5a866b2cb8611aeb7b698a20...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,False
3,Test/Rural/4194,Test,Rural,Test/Rural/images_png/4194.png,None,False,1024,1024,3,RGB,PNG,.png,None,None,None,1900496,NaN,True,False,None,NaN,NaN,NaN,NaN,NaN,None,0c4fc7c7ce38b0de06538b5c17e8b6819d454bf91440c3...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,False
4,Test/Rural/4195,Test,Rural,Test/Rural/images_png/4195.png,None,False,1024,1024,3,RGB,PNG,.png,None,None,None,2055632,NaN,True,False,None,NaN,NaN,NaN,NaN,NaN,None,a2cc9bb8daff0b047500188f3737be10d179df7e743365...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,False


,count
issue_type,


## 8–11. Análisis general, clases, Urban/Rural y splits

Se separan dos denominadores: (A) todos los píxeles de máscaras legibles y (B) solo píxeles válidos 1–7. Para cada clase se calcula cantidad, distribución, presencia y cobertura por imagen. Los mismos acumuladores se agrupan por dominio y split; el test se inspecciona realmente para comprobar si tiene ground truth local.

In [5]:
if analysis:
    print("Conteos por split")
    display(analysis["split_counts_df"])
    print("Conteos por dominio")
    display(analysis["domain_counts_df"])
    print("Estadísticas de píxeles por clase")
    display(analysis["class_pixel_counts_df"])
    print("Presencia de clases")
    display(analysis["class_image_presence_df"])

Conteos por split


,split,image_count,percentage_images,mask_count,readable_mask_count,urban_image_count,rural_image_count,ignore_pixel_count,ignore_percentage_all_pixels
0,Test,1796,29.998330,0,0,820,976,0,NaN
1,Train,2522,42.124603,2522,2522,1156,1366,99632059,3.767507
2,Val,1669,27.877067,1669,1669,677,992,45777073,2.615723


Conteos por dominio


,domain,image_count,percentage_images,mask_count,readable_mask_count,ignore_pixel_count,ignore_percentage_all_pixels
0,Rural,3334,55.687323,2358,2358,59482670,2.405729
1,Urban,2653,44.312677,1833,1833,85926462,4.470587


Estadísticas de píxeles por clase


,class_id,class_name,pixel_count,percentage_all_pixels,percentage_valid_pixels,images_present,percentage_images_present,mean_coverage_when_present_pct_valid,median_coverage_when_present_pct_valid,min_coverage_when_present_pct_valid,max_coverage_when_present_pct_valid
0,1,background,1525959012,34.723644,35.911907,4111,98.091148,36.367666,34.227467,0.000095,100.000000
1,2,building,404299316,9.199949,9.514777,3110,74.206633,12.899200,7.555389,0.000286,94.179673
2,3,road,213710339,4.863041,5.029457,3154,75.256502,6.733975,4.128551,0.000477,64.246750
3,4,water,362297559,8.244187,8.526308,3024,72.154617,11.822210,6.008816,0.000191,100.000000
4,5,barren,207131990,4.713349,4.874643,1819,43.402529,11.415579,5.624962,0.000095,99.635601
5,6,forest,534690631,12.167042,12.583405,3052,72.822715,17.356415,8.108807,0.000095,100.000000
6,7,agriculture,1001084037,22.779960,23.559504,2473,59.007397,39.931247,39.052963,0.000095,100.000000


Presencia de clases


,class_id,class_name,images_present,images_with_readable_mask,percentage_images_present
0,1,background,4111,4191,98.091148
1,2,building,3110,4191,74.206633
2,3,road,3154,4191,75.256502
3,4,water,3024,4191,72.154617
4,5,barren,1819,4191,43.402529
5,6,forest,3052,4191,72.822715
6,7,agriculture,2473,4191,59.007397


## 12. Visualización del dataset

Las figuras usan datos reales, selección reproducible, 200 dpi y cierre explícito. La paleta proviene de `Semantic_Segmentation/render.py` del repositorio oficial: ignore negro; background blanco; building rojo; road amarillo; water azul; barren púrpura; forest verde; agriculture durazno. Solo se colorean copias en memoria para visualizar; las máscaras originales permanecen intactas.

In [6]:
if analysis:
    display(analysis["figure_status_df"])
    print("Figuras creadas en:", FIGURES_DIR)

,figure,created,reason_if_omitted
0,01_random_images.png,True,None
1,02_ground_truth_overlay.png,True,None
2,03_split_distribution.png,True,None
3,04_domain_distribution.png,True,None
4,05_class_pixel_distribution.png,True,None
5,06_class_image_presence.png,True,None
6,07_urban_vs_rural_classes.png,True,None
7,08_classes_per_image.png,True,None
8,09_ignore_distribution.png,True,None
9,10_representative_examples.png,True,None


Figuras creadas en: outputs\figures


## 13. Duplicados y riesgo de data leakage

Los duplicados exactos se agrupan por SHA-256 de los bytes de la imagen. Los hashes de máscaras se conservan como contexto. Este método **no** detecta tiles similares, imágenes vecinas, solapamiento espacial ni cercanía geográfica. Si los archivos no traen metadata de ubicación explícita, esa limitación no puede resolverse solo con nombres/hashes. Un grupo duplicado deberá mantenerse íntegro en un único subconjunto futuro.

In [7]:
if analysis:
    duplicates_df = analysis["duplicates_df"]
    print(f"Filas pertenecientes a grupos duplicados: {len(duplicates_df)}")
    display(duplicates_df.head(20))

Filas pertenecientes a grupos duplicados: 0


,duplicate_group_id,image_id,split,domain,image_path,mask_path,sha256_image,sha256_mask,group_size,within_split_duplicate,cross_split_duplicate


## 14. Dataset lógico y archivos de salida

La limpieza es lógica mediante flags y DataFrames. No se mueven, renombran, reescriben ni borran imágenes o máscaras. `dataset_index.csv` es el punto de partida para filtros y splits posteriores; los restantes CSV son agregados derivados de la ejecución real.

In [8]:
if analysis:
    output_files = sorted(path.name for path in OUTPUT_DIR.glob("*.csv"))
    print("CSV generados:")
    for name in output_files:
        print("-", name)
    print()
    print("Validaciones internas:")
    display(analysis["validation_df"] )
    assert analysis["validation_df"]["passed"].all(), "Falló una validación interna; revisar antes de usar los resultados."

CSV generados:
- class_distribution_by_domain.csv
- class_distribution_by_split.csv
- class_image_presence.csv
- class_pixel_counts.csv
- dataset_index.csv
- dataset_issues.csv
- dataset_summary.csv
- domain_counts.csv
- duplicates.csv
- split_counts.csv

Validaciones internas:


,check,passed,detail
0,image_id_unique,True,Una fila por identificador de imagen.
1,ignore_index_is_zero,True,El valor ignore no fue remapeado.
2,pixel_accounting,True,Ignore más clases 1-7 coincide con el total de...
3,valid_class_percentages_sum_100,True,Suma=100.000000000000%
4,readable_masks_have_dimensions,True,Todas las máscaras legibles tienen píxeles.
5,rgb_input_channels_checked,True,Los canales se obtuvieron desde cada archivo l...
6,domain_valid_percentages_sum_100,True,Rural=100.000000000%; Urban=100.000000000%
7,split_valid_percentages_sum_100,True,Train=100.000000000%; Val=100.000000000%


## 15. Diseño experimental propuesto

No se crea ni se materializa aún ningún split nuevo. El protocolo propuesto para una entrega posterior es:

- mantener el **Test oficial** completamente intacto para evaluación externa; como no posee ground truth distribuido localmente, no puede utilizarse para calcular localmente mIoU ni Dice;
- utilizar el **Train oficial** como datos de desarrollo y dividirlo de forma reproducible en **80% de entrenamiento interno** y **20% de validación interna**;
- mantener el **Val oficial** completamente separado como conjunto de evaluación local final.

El futuro split interno usará `RANDOM_SEED` y trabajará mediante índices, sin mover archivos. Preservará aproximadamente la proporción Urban/Rural, considerará la distribución multietiqueta de clases y mantendrá juntos los eventuales duplicados exactos.

In [9]:
if analysis:
    print(experimental_design_text(dataset_index_df, analysis["duplicates_df"]))
else:
    print("Diseño condicionado a la inspección real: No disponible sin dataset.")

Splits observados: Test, Train, Val. Splits con alguna máscara local: Train, Val. Test tiene máscaras locales: no. Conservar Test oficial completamente intacto para la evaluación externa; sin su ground truth distribuido localmente no se pueden calcular mIoU ni Dice locales sobre este split. Usar Train oficial como datos de desarrollo y, en una entrega posterior, dividirlo en 80% de entrenamiento interno y 20% de validación interna. Mantener Val oficial completamente separado como conjunto de evaluación local final. La futura subdivisión será por índices (sin mover/copiar archivos), usará RANDOM_SEED, preservará aproximadamente la proporción Urban/Rural, considerará la distribución multietiqueta por presencia/cobertura de clases y mantendrá juntos los eventuales duplicados exactos. No se materializa ningún split en esta entrega. No se detectaron duplicados exactos por SHA-256, aunque este hash no detecta vecindad geográfica ni similitud visual.


## 16. Innovación propuesta: segmentación multiescala

Por el primer integrante en orden lexicográfico, Alvaro Toro Herrera, corresponde A=1 y `(1 mod 2)+1=2`: **Innovación 2 — segmentación multiescala**. En etapas posteriores se evaluará el mismo modelo sobre una imagen a distintas escalas de entrada; sus mapas volverán a una resolución común y se combinarán mediante promedio o votación. La comparación contra el método base usará exactamente los mismos datos y métricas. En esta entrega no se implementa y no se fijan escalas sin justificación experimental.

## 17. Métricas propuestas (no calculadas en Entrega 1)

Para cada clase válida $c \in \{1,\dots,7\}$:

$$IoU_c = \frac{TP_c}{TP_c + FP_c + FN_c}$$

$$mIoU = \frac{1}{|C|} \sum_{c \in C} IoU_c$$

$$Dice_c = \frac{2TP_c}{2TP_c + FP_c + FN_c}$$

Se reportarán IoU por clase, mIoU y Dice/F1 por clase con promedio macro. Su rango es 0–1 y un valor mayor es mejor. Los píxeles cuyo ground truth sea 0 se excluirán antes de formar las matrices de confusión. El promedio macro reduce el dominio de clases grandes. Pixel accuracy podrá ser secundaria, nunca la única métrica. No hay predicciones en Entrega 1, por lo que aquí no se calcula ningún resultado.

## 18. Resumen automático para el informe

El siguiente bloque imprime exclusivamente valores derivados de los DataFrames. Si el dataset no está disponible, lo declara expresamente.

In [10]:
if analysis:
    print_automatic_summary(
        dataset_index_df,
        dataset_issues_df,
        analysis["duplicates_df"],
        analysis["class_pixel_counts_df"],
    )
else:
    print("Resumen automático: No disponible; LoveDA no fue localizado.")

RESUMEN AUTOMÁTICO PARA EL INFORME
Fuente oficial: Zenodo DOI 10.5281/zenodo.5706578
Dataset: LoveDA
Número de imágenes: 5987
Número de máscaras: 4191
Splits encontrados:
- Test: 1796
- Train: 2522
- Val: 1669
Test tiene máscaras locales: no
Test sin ground truth local: 1796 imágenes (esperado)
Dominios:
- Rural: 3334
- Urban: 2653
Formatos de imagen: PNG
Formatos de máscara: PNG
Resoluciones: 1024x1024
Resolución más frecuente: 1024x1024
Canales: 3
Valores únicos observados en máscaras: [0, 1, 2, 3, 4, 5, 6, 7]
Ignore index: 0
Píxeles ignore: 145409132
Porcentaje ignore (todos los píxeles): 3.308827%
Número de clases válidas observadas: 7
Clases observadas: background, building, road, water, barren, forest, agriculture
Distribución de píxeles por clase (denominador: píxeles válidos 1-7):
- background: 1525959012 (35.911907%)
- building: 404299316 (9.514777%)
- road: 213710339 (5.029457%)
- water: 362297559 (8.526308%)
- barren: 207131990 (4.874643%)
- forest: 534690631 (12.583405%)
- 

## 19. Conclusiones del análisis inicial

Las conclusiones cuantitativas deben leerse desde el resumen anterior y los CSV, nunca desde cifras hardcodeadas. El total publicado de 5.987 imágenes se usa solo como sanity check. Las diferencias Urban/Rural son descriptivas y no implican causalidad. Los nombres de archivo pueden sugerir áreas, pero no reemplazan metadata geográfica explícita para controlar dependencia espacial. Esta entrega deja preparada la auditoría y el diseño; no selecciona arquitectura ni entrena un modelo.

In [11]:
if analysis:
    actual = len(dataset_index_df)
    published_sanity_check = 5987
    if actual == published_sanity_check:
        print(f"Sanity check: el total real indexado ({actual}) coincide con la referencia publicada.")
    else:
        print(f"Sanity check: el total real indexado ({actual}) difiere de {published_sanity_check}; no se fuerza el resultado. Revisar archivos descargados y criterios de descubrimiento.")
    print("Estado: análisis de Entrega 1 completado; modelos entrenados: 0.")
else:
    print("Estado: código preparado, ejecución cuantitativa pendiente por ausencia del dataset. Modelos entrenados: 0.")

Sanity check: el total real indexado (5987) coincide con la referencia publicada.
Estado: análisis de Entrega 1 completado; modelos entrenados: 0.
